# Evaluation, Validation & Tuning

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/T31SHA/data-ml-handbook/blob/main/notebooks/10_evaluation.ipynb)

Runnable examples from **Data/ML Handbook** — section `10_evaluation`. Read the explanations in the [docs](https://t31sha.github.io/data-ml-handbook/handbook/10_evaluation/).

In [ ]:
# Setup: works in Colab or locally from the repo root
import os, subprocess, sys
if not os.path.exists("code") and not os.path.exists("../code"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/T31SHA/data-ml-handbook.git"], check=True)
    os.chdir("data-ml-handbook")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=False)
if os.path.exists("../code"): os.chdir("..")
os.chdir("code"); os.makedirs("data", exist_ok=True)
for prep in ["proj01_make_csv.py", "s02_load.py"]:          # shared datasets used by many snippets
    if not os.path.exists("data/housing.parquet"): subprocess.run([sys.executable, prep], check=True)
sys.path.insert(0, "../figures/src")

### `code/s08_nested.py`

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
X, y = load_breast_cancer(return_X_y=True)
inner, outer = StratifiedKFold(3, shuffle=True, random_state=1), StratifiedKFold(5, shuffle=True, random_state=2)
search = GridSearchCV(make_pipeline(StandardScaler(), SVC()), {"svc__C": [0.1, 1, 10, 100], "svc__gamma": ["scale", 0.01, 0.001]}, cv=inner)
search.fit(X, y)
print(f"best inner-CV score (optimistic): {search.best_score_:.4f}  params={search.best_params_}")
nested = cross_val_score(search, X, y, cv=outer)
print(f"nested CV estimate (unbiased):    {nested.mean():.4f} ± {nested.std():.4f}")

### `code/s08_metrics.py`

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, log_loss, brier_score_loss, matthews_corrcoef
y = np.array([1, 1, 1, 1, 0, 0, 0, 0, 0, 0]); p = np.array([.9, .8, .6, .3, .7, .4, .2, .2, .1, .05])
yhat = (p >= .5).astype(int); tn, fp, fn, tp = confusion_matrix(y, yhat).ravel()
print(f"TP={tp} FP={fp} FN={fn} TN={tn}")
print(f"precision={precision_score(y, yhat):.3f} (= {tp}/{tp+fp})  recall={recall_score(y, yhat):.3f} (= {tp}/{tp+fn})  F1={f1_score(y, yhat):.3f}")
print(f"ROC-AUC={roc_auc_score(y, p):.3f}  log-loss={log_loss(y, p):.3f}  Brier={brier_score_loss(y, p):.3f}  MCC={matthews_corrcoef(y, yhat):.3f}")
pairs = [(i, j) for i in np.where(y == 1)[0] for j in np.where(y == 0)[0]]
print("AUC as P(score_pos > score_neg) =", np.mean([p[i] > p[j] for i, j in pairs]).round(3))